# Week 13 – Week 14: Ensemble Methods, Bias-Variance & Regularization

**SIT-N Hack-o-Week | 5th Semester**

### Topics Covered:
1. **Ensemble Methods**: Bagging (Random Forest) & Boosting (XGBoost, LightGBM)
2. **Bias-Variance Trade-off** and its impact on model performance
3. **Overfitting & Underfitting**: Causes, identification, and prevention
4. **Regularization (L1/L2)**: Controlling overfitting and improving generalisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed. Install with: pip install xgboost')

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM not installed. Install with: pip install lightgbm')

print('Setup complete!')

## 1. Load & Prepare Dataset

In [ ]:
df = pd.read_csv('data/kaggle_student_performance.csv')
df['Distinction'] = (df['CGPA'] >= 8.5).astype(int)

NUMERIC = ['Age', 'StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore']
CATEGORICAL = ['Gender', 'Department']

# Build preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUMERIC),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL)
])

X = df[NUMERIC + CATEGORICAL]
y = df['Distinction'].values
X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.25, random_state=42, stratify=y)
print(f'Dataset: {len(df)} students | Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Distinction distribution: {dict(zip(*np.unique(y, return_counts=True)))}')
df.head()

---
## 2. Ensemble Methods: Bagging vs Boosting

### 2.1 Bagging (Bootstrap Aggregating)
- Trains multiple models **in parallel** on random bootstrap subsets
- Reduces **variance** by averaging predictions
- Random Forest adds **feature randomisation** for further de-correlation

In [ ]:
# --- Bagging Models ---
bagging_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=50, random_state=42, n_jobs=-1
)
bagging_dt.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print('=== BAGGING RESULTS ===')
for name, model in [('Bagging (DT)', bagging_dt), ('Random Forest', rf)]:
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    print(f'{name:20s} | Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}')

### 2.2 Boosting (Sequential Learning)
- Trains models **sequentially**, each focusing on previously misclassified samples
- Reduces **bias** by iteratively correcting errors
- **XGBoost** & **LightGBM** add regularisation and histogram-based splits for speed

In [ ]:
# --- Boosting Models ---
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

boosting_models = [('Gradient Boosting', gb)]

if HAS_XGB:
    xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, eval_metric='logloss', use_label_encoder=False)
    xgb.fit(X_train, y_train)
    boosting_models.append(('XGBoost', xgb))

if HAS_LGBM:
    lgbm = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train)
    boosting_models.append(('LightGBM', lgbm))

print('=== BOOSTING RESULTS ===')
for name, model in boosting_models:
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    print(f'{name:20s} | Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}')

### 2.3 Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')
plt.bar(range(len(importances)), importances[indices], color='#10b981', alpha=0.8)
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.xticks(range(len(importances)), [f'F{i}' for i in indices])
plt.tight_layout()
plt.show()

---
## 3. Bias-Variance Trade-off

**Key Concept**: As model complexity increases:
- **Bias** decreases (model can capture more patterns)
- **Variance** increases (model becomes sensitive to training noise)
- The **optimal** model balances both, minimising total error

In [ ]:
depths = range(1, 12)
train_accs = []
test_accs = []

for d in depths:
    t_scores, v_scores = [], []
    for seed in range(5):
        Xtr, Xte, ytr, yte = train_test_split(X_processed, y, test_size=0.3, random_state=seed*10+d)
        tree = DecisionTreeClassifier(max_depth=d, random_state=42)
        tree.fit(Xtr, ytr)
        t_scores.append(accuracy_score(ytr, tree.predict(Xtr)))
        v_scores.append(accuracy_score(yte, tree.predict(Xte)))
    train_accs.append(np.mean(t_scores) * 100)
    test_accs.append(np.mean(v_scores) * 100)

best_depth = list(depths)[np.argmax(test_accs)]

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, 'o-', color='#38bdf8', label='Train Accuracy', linewidth=2)
plt.plot(depths, test_accs, 's-', color='#f59e0b', label='Test Accuracy', linewidth=2)
plt.axvline(x=best_depth, color='#10b981', linestyle='--', alpha=0.6, label=f'Optimal (depth={best_depth})')
plt.fill_between(depths, test_accs, train_accs, alpha=0.1, color='#f43f5e', label='Generalisation Gap')
plt.xlabel('Tree Depth (max_depth)', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Bias-Variance Trade-off: Tree Depth vs Accuracy', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n✅ Optimal tree depth: {best_depth} (Test Accuracy: {max(test_accs):.1f}%)')
print(f'⚠️  At depth=1 (high bias): Train={train_accs[0]:.1f}%, Test={test_accs[0]:.1f}%')
print(f'⚠️  At depth=11 (high variance): Train={train_accs[-1]:.1f}%, Test={test_accs[-1]:.1f}%, Gap={train_accs[-1]-test_accs[-1]:.1f}%')

---
## 4. Overfitting & Underfitting — Learning Curves

**Identification**:
- **Underfitting**: Both training and validation scores are LOW
- **Overfitting**: Training score is HIGH but validation score is LOW
- **Good Fit**: Both scores converge to a HIGH value

**Causes**:
- Underfitting: Model too simple, too few parameters, high bias
- Overfitting: Model too complex, too many parameters, high variance

**Prevention**:
- Add more features / increase complexity to fix underfitting
- Use regularisation, pruning, dropout, or more data to prevent overfitting
- Use cross-validation and early stopping to monitor generalisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scenarios = [
    ('Underfitting (depth=1)', DecisionTreeClassifier(max_depth=1, random_state=42), '#fbbf24'),
    ('Good Fit (RF, depth=5)', RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42), '#10b981'),
    ('Overfitting (no limit)', DecisionTreeClassifier(max_depth=None, random_state=42), '#f43f5e')
]

for ax, (title, model, color) in zip(axes, scenarios):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_processed, y, train_sizes=np.linspace(0.2, 1.0, 6), cv=cv, scoring='accuracy', n_jobs=-1
    )
    train_mean = np.mean(train_scores, axis=1) * 100
    val_mean = np.mean(val_scores, axis=1) * 100
    val_std = np.std(val_scores, axis=1) * 100
    
    ax.plot(train_sizes, train_mean, 'o-', color='#38bdf8', label='Train', linewidth=2)
    ax.plot(train_sizes, val_mean, 's-', color='#f59e0b', label='Validation', linewidth=2)
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, color='#f59e0b', alpha=0.15)
    ax.set_title(title, fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Learning Curves: Underfitting vs Good Fit vs Overfitting', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Regularization: L1 (Lasso) vs L2 (Ridge)

**Key Concepts**:
- **L1 (Lasso)**: Adds `λ·Σ|wᵢ|` penalty → drives weak coefficients to **exactly zero** → feature selection
- **L2 (Ridge)**: Adds `λ·Σwᵢ²` penalty → shrinks all coefficients **uniformly** → weight decay
- **C = 1/λ**: Inverse regularization strength. Small C → strong regularization

In [ ]:
# C-sweep: L1 vs L2
c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
results = {'l1_train': [], 'l1_test': [], 'l2_train': [], 'l2_test': [], 'l1_active': []}

for C in c_values:
    for penalty, solver in [('l1', 'liblinear'), ('l2', 'lbfgs')]:
        lr = LogisticRegression(C=C, penalty=penalty, solver=solver, random_state=42, max_iter=2000)
        lr.fit(X_train, y_train)
        results[f'{penalty}_train'].append(accuracy_score(y_train, lr.predict(X_train)) * 100)
        results[f'{penalty}_test'].append(accuracy_score(y_test, lr.predict(X_test)) * 100)
        if penalty == 'l1':
            results['l1_active'].append(np.sum(np.abs(lr.coef_[0]) > 1e-6))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Test Accuracy
ax1.semilogx(c_values, results['l1_test'], 'o-', color='#fbbf24', label='L1 (Lasso)', linewidth=2)
ax1.semilogx(c_values, results['l2_test'], 's-', color='#38bdf8', label='L2 (Ridge)', linewidth=2)
ax1.set_xlabel('C (Inverse Regularization Strength)', fontsize=11)
ax1.set_ylabel('Test Accuracy (%)', fontsize=11)
ax1.set_title('L1 vs L2: Test Accuracy across C values', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# L1 Active Features
total_feats = len(LogisticRegression(C=1, penalty='l1', solver='liblinear').fit(X_train, y_train).coef_[0])
ax2.semilogx(c_values, results['l1_active'], 'o-', color='#f43f5e', linewidth=2)
ax2.axhline(y=total_feats, color='#10b981', linestyle='--', alpha=0.5, label=f'Total Features ({total_feats})')
ax2.set_xlabel('C (Inverse Regularization Strength)', fontsize=11)
ax2.set_ylabel('Active Features (non-zero coefficients)', fontsize=11)
ax2.set_title('L1 Feature Sparsity: Active Features vs C', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.1 Coefficient Comparison (No Reg vs L1 vs L2)

In [ ]:
lr_none = LogisticRegression(penalty=None, solver='lbfgs', random_state=42, max_iter=2000).fit(X_train, y_train)
lr_l1 = LogisticRegression(C=1.0, penalty='l1', solver='liblinear', random_state=42, max_iter=2000).fit(X_train, y_train)
lr_l2 = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', random_state=42, max_iter=2000).fit(X_train, y_train)

coef_df = pd.DataFrame({
    'No Regularization': lr_none.coef_[0],
    'L1 (Lasso)': lr_l1.coef_[0],
    'L2 (Ridge)': lr_l2.coef_[0]
}, index=[f'Feature {i}' for i in range(len(lr_none.coef_[0]))])

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(coef_df))
width = 0.25

ax.bar(x - width, coef_df['No Regularization'], width, color='#94a3b8', alpha=0.7, label='No Regularization')
ax.bar(x, coef_df['L1 (Lasso)'], width, color='#fbbf24', alpha=0.8, label='L1 (Lasso)')
ax.bar(x + width, coef_df['L2 (Ridge)'], width, color='#38bdf8', alpha=0.8, label='L2 (Ridge)')

ax.axhline(y=0, color='white', linewidth=0.5, alpha=0.3)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Magnitudes: No Reg vs L1 vs L2 (C=1.0)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'F{i}' for i in range(len(coef_df))])
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'\nL1 zeroed coefficients: {np.sum(np.abs(lr_l1.coef_[0]) < 1e-6)} / {len(lr_l1.coef_[0])}')
print(f'L2 zeroed coefficients: {np.sum(np.abs(lr_l2.coef_[0]) < 1e-6)} / {len(lr_l2.coef_[0])}')
print(f'\n📌 L1 drives weak coefficients to EXACTLY ZERO → built-in feature selection')
print(f'📌 L2 shrinks all coefficients UNIFORMLY → no feature elimination')

---
## Summary

| Concept | Key Takeaway |
|---|---|
| **Bagging** | Reduces variance via parallel bootstrap models (Random Forest) |
| **Boosting** | Reduces bias via sequential error-correcting models (XGBoost, LightGBM) |
| **Bias-Variance** | Low complexity → high bias; High complexity → high variance |
| **Overfitting** | Model memorises noise; fix with regularization, pruning, more data |
| **Underfitting** | Model too simple; fix with more features, complexity |
| **L1 (Lasso)** | Sparse coefficients, feature selection |
| **L2 (Ridge)** | Uniform weight shrinkage, weight decay |